### Handwritten Digit Classification with Pytorch (MNIST Dataset)

In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# ----- Prepare the MNIST data -----
# Load training dataset
train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True, # Load training part
    transform=transforms.ToTensor(),
    download=True
)
# Load testing dataset
test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False, # Load testing part
    transform=transforms.ToTensor()
)

# Create DataLoaders to feed data to the model in batches
batch_size = 64
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# ----- Build the Neural Network model -----
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        # nn.Sequential connects layers in order
        self.model = nn.Sequential(
            nn.Flatten(), # Flatten 28x28 image into a 784-element vector
            nn.Linear(in_features=28*28, out_features=128), # First hidden layer
            nn.ReLU(), # Activation function
            nn.Linear(in_features=128, out_features=10) # Output layer
        )
    def forward(self, x):
        return self.model(x)

model = SimpleNN()

# Define loss function and optimizer
loss_function = nn.CrossEntropyLoss() # Suitable for multiclass classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ----- Training loop -----
num_epochs = 5

for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        outputs = model(images)
        loss = loss_function(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i + 1) % 200 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Step {i+1}/{len(train_loader)}, Loss: {loss.item():.4f}")

# ----- Evaluate the model on the test set -----
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        output = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item() # Count correct predictions
    
accuracy = 100 * correct / total
print(f"Model accuracy on 10000 test images: {accuracy:.2f}%")

Epoch 1/5, Step 200/938, Loss: 0.2857
Epoch 1/5, Step 400/938, Loss: 0.2631
Epoch 1/5, Step 600/938, Loss: 0.3505
Epoch 1/5, Step 800/938, Loss: 0.2223
Epoch 2/5, Step 200/938, Loss: 0.2632
Epoch 2/5, Step 400/938, Loss: 0.1593
Epoch 2/5, Step 600/938, Loss: 0.0631
Epoch 2/5, Step 800/938, Loss: 0.0697
Epoch 3/5, Step 200/938, Loss: 0.2317
Epoch 3/5, Step 400/938, Loss: 0.1634
Epoch 3/5, Step 600/938, Loss: 0.0328
Epoch 3/5, Step 800/938, Loss: 0.1263
Epoch 4/5, Step 200/938, Loss: 0.0718
Epoch 4/5, Step 400/938, Loss: 0.0406
Epoch 4/5, Step 600/938, Loss: 0.0229
Epoch 4/5, Step 800/938, Loss: 0.0276
Epoch 5/5, Step 200/938, Loss: 0.0442
Epoch 5/5, Step 400/938, Loss: 0.0709
Epoch 5/5, Step 600/938, Loss: 0.0986
Epoch 5/5, Step 800/938, Loss: 0.0865


RuntimeError: The size of tensor a (32) must match the size of tensor b (64) at non-singleton dimension 0